In [1]:
import pandas as pd
from pathlib import Path

# Initialize path
processed_dir = Path("../data/processed")

# Load datasets
belief_clean = pd.read_parquet(processed_dir / "belief_clean.parquet")
ratings_clean = pd.read_parquet(processed_dir / "ratings_clean.parquet")
recommendations_clean = pd.read_parquet(processed_dir / "recommendations_clean.parquet")

In [2]:
# Find the most recent user prediction before each recommendation event

# Keep unseen movies with a valid user prediction
belief_predictions = belief_clean[
    (belief_clean["is_seen"] == 0) &
    (belief_clean["user_predict_rating"] >= 0.5)
][
    ["user_id", "movie_id", "tstamp", "user_predict_rating"]
].copy()

# Rename timestamp to avoid confusion with recommendation timestamp
belief_predictions = belief_predictions.rename(
    columns={"tstamp": "belief_tstamp"}
)

recommendation_events = recommendations_clean[
    ["user_id", "movie_id", "tstamp", "predictedRating"]
].copy()

recommendation_events = recommendation_events.rename(
    columns={"tstamp": "recommendation_tstamp"}
)

# Create a unique ID for each recommendation event
recommendation_events["recommendation_id"] = range(
    len(recommendation_events)
)

# Standardize timestamp precision
belief_predictions["belief_tstamp"] = (
    pd.to_datetime(belief_predictions["belief_tstamp"])
    .astype("datetime64[ns]")
)

recommendation_events["recommendation_tstamp"] = (
    pd.to_datetime(recommendation_events["recommendation_tstamp"])
    .astype("datetime64[ns]")
)

# Sort for merge_asof
belief_predictions = (
    belief_predictions
    .sort_values(
        ["belief_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)

recommendation_events = (
    recommendation_events
    .sort_values(
        ["recommendation_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)


# Match the most recent belief before each recommendation
recommendation_analysis = pd.merge_asof(
    recommendation_events,
    belief_predictions,
    left_on="recommendation_tstamp",
    right_on="belief_tstamp",
    by=["user_id", "movie_id"],
    direction="backward",
    allow_exact_matches=False
)

print(f"Recommendation events: {len(recommendation_events)}")
print(f"Recommendations with prior user prediction: {recommendation_analysis['user_predict_rating'].notna().sum()}")
print(f"Recommendations without prior user prediction: {recommendation_analysis['user_predict_rating'].isna().sum()}")

Recommendation events: 1285352
Recommendations with prior user prediction: 57192
Recommendations without prior user prediction: 1228160


In [3]:
# Attribute each rating to the most recent recommendation
# for the same user-movie pair before the rating

# Keep ratings with a valid rating value
ratings_for_attribution = ratings_clean[
    ratings_clean["rating"] >= 0
][
    ["user_id", "movie_id", "tstamp", "rating"]
].copy()

ratings_for_attribution = ratings_for_attribution.rename(
    columns={"tstamp": "rating_tstamp"}
)

# Standardize timestamp precision
ratings_for_attribution["rating_tstamp"] = (
    pd.to_datetime(ratings_for_attribution["rating_tstamp"])
    .astype("datetime64[ns]")
)

recommendation_analysis["recommendation_tstamp"] = (
    pd.to_datetime(recommendation_analysis["recommendation_tstamp"])
    .astype("datetime64[ns]")
)

# Keep the recommendation rows with a valid recommendation timestamp
recommendations_for_attribution = recommendation_analysis[
    [
        "recommendation_id",
        "user_id",
        "movie_id",
        "recommendation_tstamp",
    ]
].copy()

# Sort by timestamp for merge_asof
ratings_for_attribution = (
    ratings_for_attribution
    .sort_values(
        ["rating_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)

recommendations_for_attribution = (
    recommendations_for_attribution
    .sort_values(
        ["recommendation_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)

# For each rating, find the most recent recommendation
# for the same user-movie pair
rating_attribution = pd.merge_asof(
    ratings_for_attribution,
    recommendations_for_attribution,
    left_on="rating_tstamp",
    right_on="recommendation_tstamp",
    by=["user_id", "movie_id"],
    direction="backward",
    allow_exact_matches=False
)

# A rating is attributed to a recommendation if a prior
# recommendation exists
rating_attribution["consumed_after_recommendation"] = (
    rating_attribution["recommendation_tstamp"].notna()
)

# Calculate time from recommendation to rating
rating_attribution["time_to_consumption"] = (
    rating_attribution["rating_tstamp"]
    - rating_attribution["recommendation_tstamp"]
)

print(f"Ratings attributed to a recommendation: {rating_attribution['consumed_after_recommendation'].sum():,}")
print(f"Ratings not attributed to a recommendation: {(~rating_attribution['consumed_after_recommendation']).sum():,}")

Ratings attributed to a recommendation: 9,151
Ratings not attributed to a recommendation: 1,745,464


In [4]:
# Mark recommendations that received an attributed rating
attributed_recommendations = (
    rating_attribution.loc[
        rating_attribution["consumed_after_recommendation"],
        [
            "recommendation_id",
            "rating_tstamp",
            "rating",
            "time_to_consumption",
        ],
    ]
    .sort_values(
        ["recommendation_id", "rating_tstamp"]
    )
    .drop_duplicates(
        subset=["recommendation_id"],
        keep="first",
    )
)

recommendation_analysis = recommendation_analysis.merge(
    attributed_recommendations,
    on="recommendation_id",
    how="left"
)

# Create consumption indicator
recommendation_analysis["consumed_after_recommendation"] = (
    recommendation_analysis["rating_tstamp"].notna()
)

In [5]:
# Find the most recent valid rating before each recommendation

ratings_prior = ratings_clean[
    ratings_clean["rating"] >= 0
][
    ["user_id", "movie_id", "tstamp", "rating"]
].copy()

ratings_prior = ratings_prior.rename(
    columns={
        "tstamp": "prior_rating_tstamp",
        "rating": "prior_rating"
    }
)

# Standardize timestamp precision
ratings_prior["prior_rating_tstamp"] = (
    pd.to_datetime(ratings_prior["prior_rating_tstamp"])
    .astype("datetime64[ns]")
)

recommendation_analysis["recommendation_tstamp"] = (
    pd.to_datetime(recommendation_analysis["recommendation_tstamp"])
    .astype("datetime64[ns]")
)

# Sort for merge_asof
ratings_prior = (
    ratings_prior
    .sort_values(
        ["prior_rating_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)

recommendation_analysis = (
    recommendation_analysis
    .sort_values(
        ["recommendation_tstamp", "user_id", "movie_id"]
    )
    .reset_index(drop=True)
)

# Match most recent rating before recommendation
recommendation_analysis = pd.merge_asof(
    recommendation_analysis,
    ratings_prior,
    left_on="recommendation_tstamp",
    right_on="prior_rating_tstamp",
    by=["user_id", "movie_id"],
    direction="backward",
    allow_exact_matches=False
)

# Create prior consumption flag
recommendation_analysis["prior_consumption"] = (
    recommendation_analysis["prior_rating_tstamp"].notna()
)

print(f"Recommendations with prior consumption: {recommendation_analysis['prior_consumption'].sum()}")
print(f"Recommendations without prior consumption: {(~recommendation_analysis['prior_consumption']).sum()}")

Recommendations with prior consumption: 76127
Recommendations without prior consumption: 1209225


In [6]:
recommendation_analysis.head()

,user_id,movie_id,recommendation_tstamp,predictedRating,recommendation_id,belief_tstamp,user_predict_rating,rating_tstamp,rating,time_to_consumption,consumed_after_recommendation,prior_rating_tstamp,prior_rating,prior_consumption
0,377084,296,2023-03-01 06:10:51,4.626386,0,NaT,NaN,NaT,NaN,NaT,False,2021-08-22 00:23:43,1.5,True
1,377084,924,2023-03-01 06:10:51,4.303385,2,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
2,377084,1201,2023-03-01 06:10:51,4.215998,5,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
3,377084,1204,2023-03-01 06:10:51,4.236355,7,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False
4,377084,1258,2023-03-01 06:10:51,4.241269,4,NaT,NaN,NaT,NaN,NaT,False,NaT,NaN,False


In [7]:
# Create two analysis-ready tables: 
# - Does the system align with user predictions? (expectation_analysis)
# - Is recommendation associated with subsequent consumption? (consumption_analysis)

# Expectation analysis:
# Keep recommendations for movies the user had not previously consumed
# and for which a valid user prediction exists

expectation_analysis = recommendation_analysis[
    (~recommendation_analysis["prior_consumption"]) &
    (recommendation_analysis["user_predict_rating"].notna())
].copy()

# Consumption analysis:
# Keep recommendations for movies the user had not previously consumed
consumption_analysis = recommendation_analysis[
    ~recommendation_analysis["prior_consumption"]
].copy()

In [8]:
len(expectation_analysis), len(consumption_analysis)

(55011, 1209225)

In [9]:
# Create processed data directory
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# Save analysis-ready tables
expectation_analysis.to_parquet(
    processed_dir / "expectation_analysis.parquet",
    index=False
)

consumption_analysis.to_parquet(
    processed_dir / "consumption_analysis.parquet",
    index=False
)

print("Analysis-ready datasets saved successfully.")

Analysis-ready datasets saved successfully.
